## Installing Docker Engine & post-install

For any real use, install Docker Engine from **Docker's official package repositories** — *not* the distro's default repos (often months behind) and *not* the convenience script (fine for throwaway VMs, but it skips integrity checks). The Ubuntu/Debian flow: add Docker's GPG key, add the repo, then install the five packages:

```bash
sudo apt-get install -y docker-ce docker-ce-cli containerd.io \
  docker-buildx-plugin docker-compose-plugin
```

Those five are the whole stack you've been using: **`docker-ce`** (the `dockerd` daemon), **`docker-ce-cli`** (the `docker` client), **`containerd.io`** (the runtime supervisor), and the **buildx** and **compose** plugins.

**Post-install — three adjustments make it usable:**

**1. Run `docker` without `sudo`.** The socket `/var/run/docker.sock` is owned by `root:docker`, so add yourself to the `docker` group:

```bash
sudo usermod -aG docker $USER && newgrp docker
```

**Security caveat:** the `docker` group is *effectively root* — a member can `docker run -v /:/host alpine` and own the box. On shared hosts, prefer rootless (next section).

**2. Enable on boot** so the daemon survives reboots:

```bash
sudo systemctl enable --now docker
```

**3. Verify** end to end — the three checks from module 01: `docker version` (client *and* server), `docker info` (daemon config, storage driver, kernel), and `docker run hello-world` (a full round trip).

And know **where the data lives**: everything sits under `/var/lib/docker/` — `overlay2/` (layers), `volumes/` (named volumes), `containers/<id>/` (runtime state + logs). That path is the first thing you relocate when the root partition fills, which is exactly what `daemon.json` handles next.